In [1]:
def get_training_chunks(num_services, granularity):
    if granularity is None:
        return [list(range(num_services))]
    # Create chunks of size 'granularity'
    return [list(range(i, min(i + granularity, num_services)))
            for i in range(0, num_services, granularity)]

In [2]:
from typing import List, Optional

import pandas as pd
import torch

from agent.components.commons import ServiceFeatureMapping, ServiceType
from notebooks.create_deepGP import ServiceGP, prepare_chained_data


class DynamicServiceChain(torch.nn.Module):
    def __init__(self, service_configs: List[ServiceFeatureMapping], num_inducing: int = 64):
        super().__init__()
        self.configs = service_configs
        self.gp_layers = torch.nn.ModuleList()
        self.likelihoods = torch.nn.ModuleList()

        for i, config in enumerate(service_configs):
            input_dims = len(config.feature_indices)
            if i > 0:
                input_dims += 1  # Previous service output

            gp = ServiceGP(input_dims=input_dims, num_inducing=num_inducing)
            likelihood = gpytorch.likelihoods.GaussianLikelihood()
            self.gp_layers.append(gp)
            self.likelihoods.append(likelihood)

    def forward(self, x, boundary_indices: List[int]):
        dists = []
        last_output = None

        for i, gp in enumerate(self.gp_layers):
            indices = self.configs[i].feature_indices
            current_input = x[:, indices]

            if last_output is not None:
                # DETACH logic: If 'i' is the start of a new chunk,
                # we block the gradient from flowing back to 'i-1'.
                if i in boundary_indices:
                    inp_sample = last_output.detach()
                else:
                    inp_sample = last_output

                current_input = torch.cat([current_input, inp_sample], dim=-1)

            dist = gp(current_input)
            dists.append(dist)
            last_output = dist.rsample().unsqueeze(-1)

        return tuple(dists)


In [3]:

from agent.components import RASK

raw_df = pd.read_csv("../statics/metrics_TSC_EXPLORE.csv")
converted_df = RASK.preprocess_data(raw_df)

QR_MAP = ServiceFeatureMapping(ServiceType.QR, [0, 1])
CV_MAP = ServiceFeatureMapping(ServiceType.CV, [2, 3, 4])
PC_MAP = ServiceFeatureMapping(ServiceType.PC, [5, 6])

repetitions = 2
# chain_definition = ([QR_MAP])
chain_definition = (([QR_MAP] * repetitions) + ([CV_MAP] * repetitions) + ([PC_MAP] * repetitions))# +
                    # ([QR_MAP] * repetitions) + ([CV_MAP] * repetitions) + ([PC_MAP] * repetitions) +
                    # ([QR_MAP] * repetitions) + ([CV_MAP] * repetitions) + ([PC_MAP] * repetitions))

train_loader, test_x, test_y, scaler_X = prepare_chained_data(converted_df, chain_definition, test_size=0.1)


In [4]:


def run_training(granularity: Optional[int] = None):
    # Setup
    torch.set_default_dtype(torch.float64)

    # Dummy data creation for standalone functionality
    # (In real usage, replace with your RASK.preprocess_data(raw_df))
    # data = {'max_tp': np.random.rand(90), 'cores': np.random.rand(90),
    #         'data_quality': np.random.rand(90), 'model_size': np.random.rand(90)}
    # df = pd.DataFrame(data)

    # train_loader, test_x, test_y, _ = prepare_chained_data(df, chain_definition, test_size = 0.9)
    model = DynamicServiceChain(chain_definition)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

    mlls = [
        gpytorch.mlls.VariationalELBO(model.likelihoods[i], model.gp_layers[i], num_data=len(train_loader.dataset))
        for i in range(len(chain_definition))
    ]

    # Example: granularity=2, num_services=6
    # chunks = [[0, 1], [2, 3], [4, 5]]
    # boundaries = [2, 4] (Indices where we cut the gradient)

    chunks = get_training_chunks(len(chain_definition), granularity)
    boundaries = [chunk[0] for chunk in chunks if chunk[0] != 0]

    print(f"Starting training with granularity={granularity}. Chunks: {chunks}")

    model.train()
    for epoch in range(350):
        for x_batch, y_batch in train_loader:
            optimizer.zero_grad()

            # 1. Forward pass tells the model where to break the chain
            distributions = model(x_batch, boundary_indices=boundaries)

            # 2. Backward passes for each chunk
            for chunk_indices in chunks:
                # This loss ONLY sees the graph for its specific chunk
                loss = sum([-mlls[i](distributions[i], y_batch[:, i]) for i in chunk_indices])

                # TODO: verify if this makes a difference
                # Since chunks are detached from each other, they don't share paths.
                # You likely don't even need retain_graph=True now!
                loss.backward(retain_graph=True)

            optimizer.step()

        # if epoch % 10 == 0:
        #     print(f"Epoch {epoch} | Loss: {loss.item():.4f}")
        if epoch % 50 == 0:
            for i, likelihood in enumerate(model.likelihoods):
                print(f"Noise: {likelihood.noise.item():.4f}")


    for i, gp_layer in enumerate(model.gp_layers):
        print(gp_layer.covar_module.base_kernel.lengthscale)

    return model, boundaries


In [5]:
import torch
import gpytorch


def evaluate_model_performance(dgp_model, test_x, test_y, boundaries, num_mc=100):
    """
    Evaluates a Deep GP chain of arbitrary length.

    Args:
        dgp_model: The DynamicServiceChain instance.
        test_x: Test features tensor.
        test_y: Ground truth targets tensor.
        num_mc: Number of Monte Carlo simulations.
    """
    dgp_model.eval()
    all_samples = []
    num_services = len(dgp_model.configs)

    with torch.no_grad(), gpytorch.settings.cholesky_jitter(1e-3):
        for _ in range(num_mc):
            # 1. Dynamically capture all output distributions
            distributions = dgp_model(test_x, boundary_indices=boundaries)

            # 2. Sample from each distribution and stack
            # This replaces the hard-coded [qr_d.sample(), ...] logic
            samples = [dist.sample() for dist in distributions]
            s = torch.stack(samples, dim=-1)
            all_samples.append(s)

        # Shape: (num_mc, num_test_samples, num_services)
        combined_samples = torch.stack(all_samples, dim=0).cpu().numpy()

    # --- MODEL VALIDATION ---
    # Get the mean prediction for each service across MC samples
    predicted_means = combined_samples.mean(axis=0)

    # Convert ground truth to numpy
    actual_y = test_y.cpu().numpy()

    print(f"\n--- Model Evaluation ({num_services} Services) ---")

    # 3. Calculate Error dynamically using the names in the model config
    for i, config in enumerate(dgp_model.configs):
        # Use the name or service_type from the config
        # name = config.name if config.name else config.service_type.value

        rmse = np.sqrt(np.mean((predicted_means[:, i] - actual_y[:, i]) ** 2))
        print(f"Stage {i} [{config.service_type.value}] RMSE: {rmse:.4f}")

    return combined_samples

In [6]:
import matplotlib.pyplot as plt
import numpy as np


def draw_distribution(combined_samples, chain_definition):
    # 1. Calculate the mean and std for EACH test case across simulations
    # combined_samples shape: (num_mc, num_test_samples, num_services)
    means_per_config = combined_samples.mean(axis=0)  # Shape: (N_test, N_services)
    stds_per_config = combined_samples.std(axis=0)  # Shape: (N_test, N_services)

    # 2. Average across all test configurations to get the global chain profile
    global_means = means_per_config.mean(axis=0)  # Shape: (N_services,)
    global_stds = stds_per_config.mean(axis=0)  # Shape: (N_services,)

    # Extract service names for the X-axis labels
    stages = range(len(chain_definition))

    plt.figure(figsize=(12, 6))

    # 3. Plot the Global Staircase
    # fmt='-s' connects the dots to show the throughput "decay" or bottlenecking
    plt.errorbar(stages, global_means, yerr=global_stds, fmt='-s', capsize=10,
                 color='navy', ecolor='red', elinewidth=3, markersize=8,
                 label='Average Chain Performance')

    # 4. Annotate the Average Sigma for each stage

    for i, stage in enumerate(stages):
        plt.text(i + 0.05, global_means[i], f'Avg σ = {global_stds[i]:.3f}',
                 va='bottom', fontweight='bold', color='red', fontsize=11)
    # for i, stage_name in enumerate(stages):
    #     plt.text(i + 0.1, global_means[i], f'σ={global_stds[i]:.3f}',
    #              va='bottom', fontweight='bold', color='red', fontsize=9)

    plt.title(f"Global Throughput Bottleneck ({len(stages)} Stages)")
    plt.ylabel("Average Scaled Throughput (0-1)")
    plt.xlabel("Service Pipeline Sequence")

    plt.grid(axis='y', alpha=0.3)
    plt.legend(loc='upper right')

    # 5. Shaded area showing Hardware Diversity (Best vs Worst config)
    plt.fill_between(stages,
                     means_per_config.min(axis=0),
                     means_per_config.max(axis=0),
                     color='gray', alpha=0.1, label='Hardware Spread')

    plt.tight_layout()
    plt.show()

In [ ]:
model_none, boundaries_none = run_training(granularity=None)
# combined_samples_none = evaluate_model_performance(model_none, test_x, test_y, boundaries_none)
# draw_distribution(combined_samples_none, chain_definition)

Starting training with granularity=None. Chunks: [[0, 1, 2, 3, 4, 5]]
Noise: 0.7007
Noise: 0.7008
Noise: 0.7008
Noise: 0.7007
Noise: 0.7007
Noise: 0.7007


In [ ]:
# model, boundaries = run_training(granularity=3)
# combined_samples = evaluate_model_performance(model, test_x, test_y, boundaries)
# draw_distribution(combined_samples, chain_definition)

In [ ]:
# model, boundaries = run_training(granularity=2)
# combined_samples = evaluate_model_performance(model, test_x, test_y, boundaries)
# draw_distribution(combined_samples, chain_definition)

In [ ]:
model_one, boundaries_one = run_training(granularity=1)
# combined_samples_one = evaluate_model_performance(model_one, test_x, test_y, boundaries_one)
# draw_distribution(combined_samples_one, chain_definition)

In [ ]:
raw_df = pd.read_csv("../statics/metrics_DEMO_OPERATE.csv")
converted_df = RASK.preprocess_data(raw_df)

train_loader, test_x, test_y, scaler_X = prepare_chained_data(converted_df, chain_definition, test_size=0.9)

combined_samples_one = evaluate_model_performance(model_one, test_x, test_y, boundaries_one)
draw_distribution(combined_samples_one, chain_definition)

combined_samples_none = evaluate_model_performance(model_none, test_x, test_y, boundaries_none)
draw_distribution(combined_samples_none, chain_definition)

In [ ]:
def plot_residual_divergence(samples_none, samples_one, test_y):
    # Calculate Error (Predicted Mean - Actual)
    err_none = np.abs(samples_none.mean(axis=0) - test_y.cpu().numpy())
    err_one = np.abs(samples_one.mean(axis=0) - test_y.cpu().numpy())

    plt.figure(figsize=(10, 5))
    plt.plot(err_none.mean(axis=0), label='Global Error', color='blue', marker='o')
    plt.plot(err_one.mean(axis=0), label='Independent Error', color='green', marker='x')

    plt.title("Error Propagation: Global vs Independent")
    plt.xlabel("Chain Stage")
    plt.ylabel("Mean Absolute Error")
    plt.legend()
    plt.show()

plot_residual_divergence(combined_samples_none, combined_samples_one, test_y)